In [11]:
import requests
import pandas as pd
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [12]:
WAZUH_IP = "https://192.168.100.47:55000"

USERNAME = "ai_user"
PASSWORD = "Fyp@112233"

In [13]:
auth = requests.post(
    f"{WAZUH_IP}/security/user/authenticate",
    auth=(USERNAME, PASSWORD),
    verify=False
)

res = auth.json()

if "data" in res:
    TOKEN = res["data"]["token"]
    print("✅ Token received")
else:
    print("❌ Auth error:", res)

✅ Token received


In [14]:
headers = {
    "Authorization": f"Bearer {TOKEN}"
}

response = requests.get(
    f"{WAZUH_IP}/manager/logs?limit=50",
    headers=headers,
    verify=False
)

res = response.json()

if "data" in res:
    logs = res["data"]["affected_items"]
    print(f"✅ Logs fetched: {len(logs)}")
else:
    print("❌ Error:", res)
    logs = []

✅ Logs fetched: 50


In [20]:
logs.append({
    "description": "critical error multiple failed login unauthorized ddos attack flood detected",
    "level": "error"
})

In [15]:
def fusion_decision(network, cloud, host):
    score = 0

    if network != "Benign":
        score += 60
    if cloud == "Suspicious":
        score += 20
    if host == -1:
        score += 20

    if score >= 70:
        return score, "HIGH"
    elif score >= 40:
        return score, "MEDIUM"
    elif score >= 20:
        return score, "LOW"
    return score, "NORMAL"

In [21]:
results = []

for log in logs:
    msg = log.get("description", "").lower()
    level = log.get("level", "").lower()

    # CLOUD
    if level in ["error", "warning"]:
        cloud_pred = "Suspicious"
    else:
        cloud_pred = "Normal"

    # HOST
    if any(word in msg for word in ["failed", "denied", "unauthorized", "invalid", "error"]):
        host_pred = -1
    else:
        host_pred = 1

    # NETWORK
    if any(word in msg for word in ["attack", "flood", "ddos", "scan", "malware"]):
        network_pred = "DDoS"
    else:
        network_pred = "Benign"

    # FUSION
    score, alert = fusion_decision(network_pred, cloud_pred, host_pred)

    results.append({
        "message": msg,
        "network": network_pred,
        "cloud": cloud_pred,
        "host": host_pred,
        "score": score,
        "alert": alert
    })

df = pd.DataFrame(results)
df.head()

,message,network,cloud,host,score,alert
0,ending rootcheck scan.,DDoS,Normal,1,60,MEDIUM
1,evaluation finished for policy '/var/ossec/ru...,Benign,Normal,1,0,NORMAL
2,security configuration assessment scan finish...,DDoS,Normal,1,60,MEDIUM
3,(6009): file integrity monitoring scan ended.,DDoS,Normal,1,60,MEDIUM
4,fim sync module started.,Benign,Normal,1,0,NORMAL


In [22]:
print("🔍 Alert Distribution:\n")
print(df["alert"].value_counts())

🔍 Alert Distribution:

alert
NORMAL    44
MEDIUM     6
HIGH       1
Name: count, dtype: int64


In [23]:
df[df["alert"] == "HIGH"]

,message,network,cloud,host,score,alert
50,critical error multiple failed login unauthori...,DDoS,Suspicious,-1,100,HIGH


In [24]:
df.to_csv("../outputs/wazuh_ai_results.csv", index=False)

print("✅ Results saved")

✅ Results saved
